In [2]:
import os
import re
import time
import requests
import pandas as pd

stock_codes = ["000002", "000006"]
start_year = 2020
end_year = 2024

save_dir = "annual_reports"
os.makedirs(save_dir, exist_ok=True)

query_url = "http://www.cninfo.com.cn/new/hisAnnouncement/query"
download_base = "http://static.cninfo.com.cn/"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Referer": "http://www.cninfo.com.cn/new/commonUrl/pageOfSearch?url=disclosure/list/search",
    "X-Requested-With": "XMLHttpRequest"
}

def clean_title(title):
    return re.sub(r"<.*?>", "", title)

all_records = []

for code in stock_codes:
    for year in range(start_year, end_year + 1):
        print(f"\nSearching {code} {year}...")

        params = {
            "pageNum": 1,
            "pageSize": 30,
            "column": "sse" if code.startswith("6") else "szse",
            "tabName": "fulltext",
            "plate": "sh" if code.startswith("6") else "sz",
            "stock": code,
            "searchkey": "年度报告",
            "category": "category_ndbg_szsh;",
            "seDate": f"{year}-01-01~{year}-12-31",
            "isHLtitle": "true"
        }

        try:
            r = requests.post(query_url, headers=headers, data=params, timeout=20)
            print("Status:", r.status_code)

            data = r.json()
            announcements = data.get("announcements") or []

            print("Found:", len(announcements))

            for ann in announcements:
                title = clean_title(ann.get("announcementTitle", ""))
                pdf_path = ann.get("adjunctUrl", "")

                if not pdf_path:
                    continue

                if "年度报告" in title and "摘要" not in title and "英文" not in title:
                    pdf_url = download_base + pdf_path

                    filename = f"{code}_{year}_{title}.pdf"
                    filename = re.sub(r'[\\/:*?"<>|]', "_", filename)
                    filepath = os.path.join(save_dir, filename)

                    pdf = requests.get(pdf_url, headers=headers, timeout=30)

                    if pdf.status_code == 200:
                        with open(filepath, "wb") as f:
                            f.write(pdf.content)

                        print("Downloaded:", filename)

                        all_records.append({
                            "code": code,
                            "year": year,
                            "title": title,
                            "pdf_url": pdf_url,
                            "file_path": filepath
                        })
                    else:
                        print("PDF failed:", pdf.status_code)

                    time.sleep(1)

        except Exception as e:
            print(f"Error for {code} {year}: {e}")

        time.sleep(1)

records = pd.DataFrame(all_records)
records.to_csv("annual_report_download_records.csv", index=False, encoding="utf-8-sig")

print("\nDone.")
print(records.head())


Searching 000002 2020...
Status: 200
Found: 0

Searching 000002 2021...
Status: 200
Found: 0

Searching 000002 2022...
Status: 200
Found: 0

Searching 000002 2023...
Status: 200
Found: 0

Searching 000002 2024...
Status: 200
Found: 0

Searching 000006 2020...
Status: 200
Found: 0

Searching 000006 2021...
Status: 200
Found: 0

Searching 000006 2022...
Status: 200
Found: 0

Searching 000006 2023...
Status: 200
Found: 0

Searching 000006 2024...
Status: 200
Found: 0

Done.
Empty DataFrame
Columns: []
Index: []


In [3]:
stock_codes = ["000001", "000002", "600519"]
start_year = 2023
end_year = 2023

In [3]:
import pandas as pd

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/人工智能词频统计表(联表查询)105548241/AI_WordFreSta(Merge Query).xlsx"

df = pd.read_excel(file_path, engine="openpyxl")

print("所有列：")
print(df.columns)

print("\n看看前几行：")
print(df.head())

# 👉 先试 Symbol
codes = df["AI_WordFreSta.Symbol"]

print("\n这一列数据：")
print(codes.head())

# 去空值
codes = codes.dropna()

# 转字符串 + 补0
codes = codes.astype(str).str.replace(".0", "", regex=False).str.zfill(6)

# 去重
unique_codes = codes.drop_duplicates()

print("\n唯一代码数量：", len(unique_codes))
print(unique_codes.head())

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


所有列：
Index(['AI_WordFreSta.EndDate', 'AI_WordFreSta.InstitutionID',
       'AI_WordFreSta.Symbol', 'AI_WordFreSta.ShortName',
       'AI_WordFreSta.AnnualReportAIWordFre', 'AI_WordFreSta.AnnualReportWord',
       'AI_WordFreSta.MDAAIWordFre', 'AI_WordFreSta.MDAWord',
       'csmar_listedcoinfo.Nnindcd', 'csmar_listedcoinfo.Nnindnme'],
      dtype='str')

看看前几行：
  AI_WordFreSta.EndDate AI_WordFreSta.InstitutionID AI_WordFreSta.Symbol  \
0                统计截止日期                      上市公司ID                 证券代码   
1                   NaN                         NaN                  NaN   
2            2024-12-31                      101775               000002   
3            2023-12-31                      101775               000002   
4            2022-12-31                      101775               000002   

  AI_WordFreSta.ShortName AI_WordFreSta.AnnualReportAIWordFre  \
0                    证券简称                           年报人工智能总词频   
1                     NaN                        

In [7]:
code_col = "AI_WordFreSta.Symbol"

codes = df[code_col].dropna().astype(str)
codes = codes.str.replace(".0", "", regex=False).str.strip()

unique_codes = codes.drop_duplicates()

# 去掉表头行
unique_codes = unique_codes[unique_codes != "证券代码"]

df_unique = pd.DataFrame(unique_codes, columns=["证券代码"])

output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/unique_stock_codes.xlsx"
df_unique.to_excel(output_path, index=False)

print("完成！数量：", len(df_unique))
print(df_unique.head())

完成！数量： 0
Empty DataFrame
Columns: [证券代码]
Index: []


In [9]:
import pandas as pd

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/人工智能词频统计表(联表查询)105548241/AI_WordFreSta(Merge Query).xlsx"

df = pd.read_excel(file_path, engine="openpyxl")

code_col = "AI_WordFreSta.Symbol"

codes = df[code_col].dropna().astype(str)
codes = codes.str.replace(".0", "", regex=False).str.strip()

# 去掉表头行
codes = codes[codes != "证券代码"]

# 只保留6位代码
codes = codes[codes.str.len() == 6]

# 去重
df_unique = pd.DataFrame(codes.drop_duplicates().reset_index(drop=True), columns=["证券代码"])

output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/unique_stock_codes.xlsx"
df_unique.to_excel(output_path, index=False)

print("完成！数量：", len(df_unique))
print(df_unique.head())

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


完成！数量： 0
Empty DataFrame
Columns: [证券代码]
Index: []


In [10]:
import pandas as pd

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/人工智能词频统计表(联表查询)105548241/AI_WordFreSta(Merge Query).xlsx"

df = pd.read_excel(file_path, engine="openpyxl")

code_col = "AI_WordFreSta.Symbol"

codes = df[code_col]

# 1️⃣ 去空
codes = codes.dropna()

# 2️⃣ 转字符串
codes = codes.astype(str)

# 3️⃣ 清洗（关键）
codes = codes.str.strip()                 # 去空格
codes = codes.str.replace(".0", "", regex=False)

# 4️⃣ 去掉表头
codes = codes[codes != "证券代码"]

# ❗先不要筛6位，先看看数据
print("原始数据示例：")
print(codes.head(10))

# 5️⃣ 去重
unique_codes = codes.drop_duplicates()

print("数量：", len(unique_codes))

# 6️⃣ 再筛6位（最后做）
unique_codes = unique_codes[unique_codes.str.len() == 6]

print("筛完后数量：", len(unique_codes))

# 7️⃣ 导出
df_unique = pd.DataFrame(unique_codes, columns=["证券代码"])

output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/unique_stock_codes.xlsx"
df_unique.to_excel(output_path, index=False)

print("完成！")
print(df_unique.head())

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


原始数据示例：
2     000002
3     000002
4     000002
5     000002
6     000002
7     000006
8     000006
9     000006
10    000006
11    000006
Name: AI_WordFreSta.Symbol, dtype: str
数量： 4972
筛完后数量： 4972
完成！
Empty DataFrame
Columns: [证券代码]
Index: []


In [11]:
import pandas as pd

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/人工智能词频统计表(联表查询)105548241/AI_WordFreSta(Merge Query).xlsx"

df = pd.read_excel(file_path, engine="openpyxl")

code_col = "AI_WordFreSta.Symbol"

codes = df[code_col].dropna().astype(str)
codes = codes.str.strip().str.replace(".0", "", regex=False)

codes = codes[codes != "证券代码"]
codes = codes[codes.str.len() == 6]

unique_codes = codes.drop_duplicates().reset_index(drop=True)

df_unique = pd.DataFrame({
    "证券代码": unique_codes
})

output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/unique_stock_codes.xlsx"
df_unique.to_excel(output_path, index=False)

print("完成！唯一证券代码数量：", len(df_unique))
print(df_unique.head(20))
print("文件保存到：", output_path)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


完成！唯一证券代码数量： 4972
      证券代码
0   000002
1   000006
2   000007
3   000008
4   000009
5   000011
6   000012
7   000014
8   000017
9   000019
10  000020
11  000021
12  000025
13  000026
14  000027
15  000028
16  000029
17  000030
18  000031
19  000032
文件保存到： /Users/snowiiy/Desktop/26 spring/MGS3001/unique_stock_codes.xlsx


In [6]:
import os
import re
import time
import requests
import pandas as pd

# =========================
# 1. 你只改这里
# =========================
stock_codes = ["000002", "000006", "000007"]
start_year = 2020
end_year = 2024

# 手动写入 orgId，避免自动搜索接口报错
org_map = {
    "000002": "gssz0000002",   # 万科A
    "000006": "gssz0000006",   # 深振业A
    "000007": "gssz0000007",   # 浦发银行
   
}

save_dir = "annual_reports"
os.makedirs(save_dir, exist_ok=True)

# =========================
# 2. 固定设置，不要改
# =========================
query_url = "http://www.cninfo.com.cn/new/hisAnnouncement/query"
download_base = "http://static.cninfo.com.cn/"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "http://www.cninfo.com.cn/",
    "X-Requested-With": "XMLHttpRequest"
}

def clean_title(title):
    title = re.sub(r"<.*?>", "", str(title))
    title = re.sub(r'[\\/:*?"<>|]', "_", title)
    return title

def get_market_info(code):
    """
    根据股票代码判断交易所和板块
    """
    if code.startswith(("0", "3")):
        return "szse", "sz"
    elif code.startswith("6"):
        return "sse", "sh"
    else:
        return "", ""

all_records = []

# =========================
# 3. 主程序：搜索并下载年报
# =========================
for code in stock_codes:
    org_id = org_map.get(code)

    if not org_id:
        print(f"⚠️ No orgId for {code}. Please add it to org_map.")
        continue

    column, plate = get_market_info(code)
    stock_param = f"{code},{org_id}"

    for year in range(start_year, end_year + 1):
        print(f"\nSearching {code} {year}...")

        params = {
            "pageNum": 1,
            "pageSize": 30,
            "column": column,
            "tabName": "fulltext",
            "plate": plate,
            "stock": stock_param,
            "searchkey": "",
            "secid": "",
            "category": "category_ndbg_szsh;",
            "trade": "",
            "seDate": f"{year}-01-01~{year}-12-31",
            "sortName": "",
            "sortType": "",
            "isHLtitle": "true"
        }

        try:
            response = requests.post(
                query_url,
                headers=headers,
                data=params,
                timeout=20
            )

            print("Status:", response.status_code)

            if response.status_code != 200:
                print("Request failed.")
                print(response.text[:300])
                continue

            try:
                data = response.json()
            except Exception:
                print("Cannot parse JSON.")
                print(response.text[:500])
                continue

            announcements = data.get("announcements") or []
            print("Found:", len(announcements))

            if len(announcements) == 0:
                print("No annual report found for this firm-year.")
                continue

            for ann in announcements:
                title = clean_title(ann.get("announcementTitle", ""))
                adjunct_url = ann.get("adjunctUrl", "")

                if not adjunct_url:
                    continue

                # 保留年度报告，排除摘要、英文版、取消、更正等
                if (
                    "年度报告" in title
                    and "摘要" not in title
                    and "英文" not in title
                    and "取消" not in title
                    and "更正" not in title
                    and "修订" not in title
                ):
                    pdf_url = download_base + adjunct_url
                    file_name = f"{code}_{year}_{title}.pdf"
                    file_path = os.path.join(save_dir, file_name)

                    print("Downloading:", title)

                    pdf_response = requests.get(
                        pdf_url,
                        headers=headers,
                        timeout=30
                    )

                    if pdf_response.status_code == 200:
                        with open(file_path, "wb") as f:
                            f.write(pdf_response.content)

                        print("✅ Downloaded:", file_name)

                        all_records.append({
                            "code": code,
                            "year": year,
                            "title": title,
                            "pdf_url": pdf_url,
                            "file_path": file_path
                        })
                    else:
                        print("❌ PDF download failed:", pdf_response.status_code)

                    time.sleep(1)

        except Exception as e:
            print(f"❌ Error for {code} {year}: {e}")

        time.sleep(1)

# =========================
# 4. 保存下载记录
# =========================
records = pd.DataFrame(all_records)

if len(records) > 0:
    records.to_csv(
        "annual_report_download_records.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("\nDone. Records saved.")
    print(records.head())
else:
    print("\nDone, but no reports downloaded.")


Searching 000002 2020...
Status: 200
Found: 2
Downloading: 2019年年度报告
✅ Downloaded: 000002_2020_2019年年度报告.pdf

Searching 000002 2021...
Status: 200
Found: 2
Downloading: 2020年年度报告
✅ Downloaded: 000002_2021_2020年年度报告.pdf

Searching 000002 2022...
Status: 200
Found: 2
Downloading: 2021年年度报告
✅ Downloaded: 000002_2022_2021年年度报告.pdf

Searching 000002 2023...
Status: 200
Found: 2
Downloading: 2022年年度报告
✅ Downloaded: 000002_2023_2022年年度报告.pdf

Searching 000002 2024...
Status: 200
Found: 2
Downloading: 2023年年度报告
✅ Downloaded: 000002_2024_2023年年度报告.pdf

Searching 000006 2020...
Status: 200
Found: 2
Downloading: 2019年年度报告
✅ Downloaded: 000006_2020_2019年年度报告.pdf

Searching 000006 2021...
Status: 200
Found: 2
Downloading: 2020年年度报告
✅ Downloaded: 000006_2021_2020年年度报告.pdf

Searching 000006 2022...
Status: 200
Found: 2
Downloading: 2021年年度报告
✅ Downloaded: 000006_2022_2021年年度报告.pdf

Searching 000006 2023...
Status: 200
Found: 2
Downloading: 2022年年度报告
✅ Downloaded: 000006_2023_2022年年度报告.pdf

Searching

In [17]:
import os
import re
import pandas as pd
import pdfplumber

# =========================
# 1. 路径设置
# =========================
pdf_folder = "/Users/snowiiy/Desktop/26 spring/MGS3001/annual_reports"

sentence_output = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences.xlsx"
firm_year_output = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_firm_year.xlsx"

# =========================
# 2. AI关键词
# =========================
ai_keywords = [
    "人工智能", "AI", "AIGC", "生成式人工智能", "生成式AI",
    "机器学习", "深度学习", "神经网络", "大模型", "语言模型",
    "自然语言处理", "NLP", "计算机视觉", "机器视觉", "知识图谱",
    "智能化", "智慧化", "智能平台", "智能系统", "智能识别",
    "智能分析", "智能决策", "智能算法", "算法", "推荐算法",
    "预测模型", "数据模型", "数据挖掘", "数据驱动",
    "自动化", "自动识别", "自动生成", "自动控制",
    "云计算", "边缘计算", "算力", "GPU",
    "ChatGPT", "GPT", "多模态", "智能客服", "智能机器人", "机器人"
]

# =========================
# 3. 工具函数
# =========================
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                try:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                except:
                    continue
    except Exception as e:
        print(f"无法读取PDF: {pdf_path}")
        print(e)
    return text


def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", "", text)
    return text


def split_sentences(text):
    text = clean_text(text)
    sentences = re.split(r"[。！？!?；;]", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 8]
    return sentences


def find_keywords(sentence, keywords):
    sentence_lower = sentence.lower()
    matched = [kw for kw in keywords if kw.lower() in sentence_lower]
    return matched


def extract_code_year(file_name):
    # 文件名例子：000002_2023_2022年年度报告.pdf
    parts = file_name.split("_")
    code = parts[0] if len(parts) > 0 else ""
    disclosure_year = parts[1] if len(parts) > 1 else ""

    year_match = re.search(r"(\d{4})年年度报告", file_name)
    report_year = year_match.group(1) if year_match else disclosure_year

    return code, disclosure_year, report_year

# =========================
# 4. 提取AI句子
# =========================
records = []

pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]

print("准备处理PDF数量:", len(pdf_files))

for idx, file_name in enumerate(pdf_files, start=1):
    print(f"\nProcessing {idx}/{len(pdf_files)}: {file_name}")

    file_path = os.path.join(pdf_folder, file_name)

    code, disclosure_year, report_year = extract_code_year(file_name)

    text = extract_text_from_pdf(file_path)

    if len(text) < 100:
        print("文本太少，可能是扫描版PDF，跳过")
        continue

    sentences = split_sentences(text)
    total_sentences = len(sentences)

    ai_count = 0

    for sentence_id, sentence in enumerate(sentences, start=1):
        matched_keywords = find_keywords(sentence, ai_keywords)

        if matched_keywords:
            ai_count += 1

            records.append({
                "code": code,
                "disclosure_year": disclosure_year,
                "report_year": report_year,
                "file_name": file_name,
                "sentence_id": sentence_id,
                "AI_sentence": sentence,
                "matched_keywords": ",".join(matched_keywords),
                "total_sentences_in_report": total_sentences
            })

    print("AI sentences found:", ai_count)

    # 每处理一个PDF就保存一次，防止中途暂停后全部丢失
    temp_df = pd.DataFrame(records)
    temp_df.to_excel(sentence_output, index=False)

# =========================
# 5. 保存句子级数据
# =========================
df_ai = pd.DataFrame(records)
df_ai.to_excel(sentence_output, index=False)

print("\n句子级AI Disclosure文件已保存：")
print(sentence_output)
print("AI句子数量:", len(df_ai))

# =========================
# 6. 生成 firm-year 层面的 AI Disclosure
# =========================
if len(df_ai) > 0:
    firm_year = (
        df_ai.groupby(["code", "report_year"])
        .agg(
            ai_sentence_count=("AI_sentence", "count"),
            total_sentences_in_report=("total_sentences_in_report", "max")
        )
        .reset_index()
    )

    firm_year["AI_Disclosure"] = (
        firm_year["ai_sentence_count"] / firm_year["total_sentences_in_report"]
    )

    firm_year.to_excel(firm_year_output, index=False)

    print("\n公司-年份级AI Disclosure文件已保存：")
    print(firm_year_output)
    print(firm_year.head())

else:
    print("没有提取到AI句子，未生成firm-year文件。")

准备处理PDF数量: 483

Processing 1/483: 000501_2023_2022年年度报告.pdf
AI sentences found: 1

Processing 2/483: 000011_2023_2022年年度报告.pdf
AI sentences found: 6

Processing 3/483: 000402_2021_2020年年度报告.pdf
AI sentences found: 0

Processing 4/483: 000012_2024_2023年年度报告.pdf
AI sentences found: 8

Processing 5/483: 000509_2022_2021年年度报告（更新后）.pdf
AI sentences found: 0

Processing 6/483: 000411_2022_2021年年度报告.pdf
AI sentences found: 5

Processing 7/483: 600519_2023_贵州茅台2022年年度报告.pdf
AI sentences found: 2

Processing 8/483: 000039_2022_2021年年度报告.pdf
AI sentences found: 44

Processing 9/483: 000027_2021_2020年年度报告.pdf
AI sentences found: 1

Processing 10/483: 000524_2022_2021年年度报告.pdf
AI sentences found: 19

Processing 11/483: 000034_2022_2021年年度报告.pdf
AI sentences found: 102

Processing 12/483: 000061_2024_2023年年度报告.pdf
AI sentences found: 4

Processing 13/483: 000401_2022_2021年年度报告.pdf
AI sentences found: 6

Processing 14/483: 000002_2024_2023年年度报告.pdf
AI sentences found: 12

Processing 15/483: 000062_2

/var/folders/6f/q_xr3_ys4y5b78sztpmq5__r0000gn/T/ipykernel_78559/3421218489.py:127: UserWarning: Cell contents too long (35744), truncated to 32767 characters
  temp_df.to_excel(sentence_output, index=False)



Processing 134/483: 000031_2022_2021年年度报告.pdf
AI sentences found: 6

Processing 135/483: 000099_2022_2021年年度报告.pdf
AI sentences found: 3

Processing 136/483: 000498_2020_2019年年度报告.pdf
AI sentences found: 1

Processing 137/483: 000019_2023_2022年年度报告.pdf
AI sentences found: 6

Processing 138/483: 000017_2024_2023年年度报告.pdf
AI sentences found: 2

Processing 139/483: 000507_2024_2023年年度报告.pdf
AI sentences found: 2

Processing 140/483: 000509_2023_2022年年度报告.pdf
AI sentences found: 3

Processing 141/483: 000419_2022_2021年年度报告.pdf
AI sentences found: 0

Processing 142/483: 000407_2021_2020年年度报告.pdf
AI sentences found: 2

Processing 143/483: 000014_2023_2022年年度报告.pdf
AI sentences found: 1

Processing 144/483: 000505_2020_2019年年度报告.pdf
AI sentences found: 2

Processing 145/483: 000028_2020_2019年年度报告.pdf
AI sentences found: 2

Processing 146/483: 000157_2022_2021年年度报告.pdf
AI sentences found: 37

Processing 147/483: 000027_2024_2023年年度报告.pdf
AI sentences found: 11

Processing 148/483: 000029_2023

/var/folders/6f/q_xr3_ys4y5b78sztpmq5__r0000gn/T/ipykernel_78559/3421218489.py:133: UserWarning: Cell contents too long (35744), truncated to 32767 characters
  df_ai.to_excel(sentence_output, index=False)



句子级AI Disclosure文件已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences.xlsx
AI句子数量: 5337

公司-年份级AI Disclosure文件已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_firm_year.xlsx
     code report_year  ai_sentence_count  total_sentences_in_report  \
0  000001        2022                 90                       2335   
1  000002        2019                  7                       1809   
2  000002        2020                  9                       1949   
3  000002        2021                 18                       1969   
4  000002        2022                 11                       1870   

   AI_Disclosure  
0       0.038544  
1       0.003870  
2       0.004618  
3       0.009142  
4       0.005882  


In [36]:
import pandas as pd
import re

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

input_path = f"{base}/假设2数据/ai_disclosure_sentences_FIXED.xlsx"
output_path = f"{base}/假设2数据/ai_disclosure_sentences_AI_checked.xlsx"

# 读取时保持 code 为文本
df = pd.read_excel(input_path, dtype={"code": str})

# 保留原始列顺序
original_columns = df.columns.tolist()

# 修复 code，保证仍然是6位
if "code" in df.columns:
    df["code"] = (
        df["code"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.zfill(6)
    )

sentence_col = "AI_sentence"
keyword_col = "matched_keywords"

if sentence_col not in df.columns:
    raise ValueError("找不到 AI_sentence 列")

if keyword_col not in df.columns:
    raise ValueError("找不到 matched_keywords 列")

df[sentence_col] = df[sentence_col].astype(str)
df[keyword_col] = df[keyword_col].astype(str)

# 独立 AI：前后不能是英文字母
# 保留 AI技术、AI 应用、(AI)、AI，等
# 删除 Asia / NewbridgeAsia / said / chair 这种
def has_real_ai(text):
    return bool(re.search(r"(?<![A-Za-z])AI(?![A-Za-z])", text))

def keep_row(row):
    text = row[sentence_col]
    matched = row[keyword_col]

    kws = [k.strip() for k in matched.split(",") if k.strip()]

    # 如果这句话不是因为 AI 这个关键词进来的，直接保留
    if "AI" not in kws:
        return True

    # 如果 matched_keywords 里面除了 AI 还有其他关键词，保留
    # 例如 matched_keywords = "AI,智能化"
    other_kws = [k for k in kws if k != "AI"]
    if len(other_kws) > 0:
        return True

    # 如果它只匹配了 AI，就检查句子里 AI 是否是独立词
    if has_real_ai(text):
        return True

    # 否则删除：比如 NewbridgeAsia
    return False

before = len(df)
df_clean = df[df.apply(keep_row, axis=1)].copy()
after = len(df_clean)

# 保持原始列顺序
df_clean = df_clean[original_columns]

df_clean.to_excel(output_path, index=False)

print("原始行数：", before)
print("清洗后行数：", after)
print("删除行数：", before - after)
print("保存位置：", output_path)

原始行数： 5431
清洗后行数： 4687
删除行数： 744
保存位置： /Users/snowiiy/Desktop/26 spring/MGS3001/假设2数据/ai_disclosure_sentences_AI_checked.xlsx


In [37]:
import pandas as pd

# 你的文件路径
file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设2数据/ai_disclosure_sentences_AI_checked.xlsx"

# 读取 Excel
df = pd.read_excel(file_path)

# 输出路径（自动存在同一个文件夹）
output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设2数据/ai_disclosure_sentences_AI_checked.csv"

# 保存为 CSV
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("转换完成 ✅ 文件在：", output_path)

转换完成 ✅ 文件在： /Users/snowiiy/Desktop/26 spring/MGS3001/假设2数据/ai_disclosure_sentences_AI_checked.csv


In [54]:
import pandas as pd

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据/TRD_Year.xlsx"
df = pd.read_excel(file_path)

# ✅ 清理列名空格
df.columns = df.columns.str.strip()

# ✅ 转换为数值（关键！）
df["Ynshrtrd"] = pd.to_numeric(df["Ynshrtrd"], errors="coerce")
df["Ysmvosd"] = pd.to_numeric(df["Ysmvosd"], errors="coerce")

# ✅ 单位转换
df["Ysmvosd_yuan"] = df["Ysmvosd"] * 1000

# ✅ 计算 turnover
df["turnover"] = df["Ynshrtrd"] / df["Ysmvosd_yuan"]

# ✅ 处理异常值
df["turnover"] = df["turnover"].replace([float("inf"), -float("inf")], None)

# ✅ 删除前两行中文说明行
df_clean = df.iloc[2:].copy()

# ✅ 保存新文件
output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据/TRD_with_turnover.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ 新文件已生成：")
print(output_path)

print(df_clean[["Stkcd", "Trdynt", "turnover"]].head())

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ 新文件已生成：
/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据/TRD_with_turnover.csv
    Stkcd Trdynt  turnover
2  000002   2020  0.070691
3  000002   2021  0.110164
4  000002   2022  0.147576
5  000002   2023  0.177855
6  000002   2024  0.691726


In [58]:
import pandas as pd

# 路径（你自己确认一下是不是这个）
file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据/TRD_with_turnover.csv"

# 读取
df = pd.read_csv(file_path, encoding="utf-8-sig")

# 看一下列名（建议先跑）
print("原始列名：")
print(df.columns)

# 要删除的列（存在才删）
cols_to_drop = [
    "investor_sentiment_proxy_log_trading_volume",
    "annual_return_no_cash_dividend"
]

for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"已删除: {col}")

# 覆盖原文件
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print("✅ 已覆盖原文件（错误Y已删除）")

原始列名：
Index(['Stkcd', 'Trdynt', 'Ynshrtrd', 'Ysmvosd', 'Ysmvttl', 'Yretnd',
       'Ysmvosd_yuan', 'turnover'],
      dtype='str')
✅ 已覆盖原文件（错误Y已删除）


In [60]:
import pandas as pd

# 路径（你改成你自己的H1文件路径）
file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据/final_dataset_with_controls.csv"

# 读取数据
df = pd.read_csv(file_path, encoding="utf-8-sig")

# 清理列名（防止隐藏空格）
df.columns = df.columns.str.strip()

# 要删除的列
cols_to_drop = [
    "annual_report_total_words",
    "annual_report_ai_word_freq",
    "ai_disclosure_word_ratio",
    "ai_sentence_per_1000_sentences",
    "ai_word_per_1000_words",
    "annual_return_no_cash_dividend",
    "investor_sentiment_proxy_log_trading_volume"
]

# 只删除存在的列（防止报错）
cols_exist = [col for col in cols_to_drop if col in df.columns]

df = df.drop(columns=cols_exist)

print("已删除列：", cols_exist)

# ✅ 生成新文件（不会覆盖原文件）
output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/Data/H1_cleaned.csv"

df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ 新文件已生成：")
print(output_path)

已删除列： ['annual_report_total_words', 'annual_report_ai_word_freq', 'ai_disclosure_word_ratio', 'ai_sentence_per_1000_sentences', 'ai_word_per_1000_words', 'annual_return_no_cash_dividend', 'investor_sentiment_proxy_log_trading_volume']
✅ 新文件已生成：
/Users/snowiiy/Desktop/26 spring/MGS3001/Data/H1_cleaned.csv


In [61]:
import pandas as pd

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

# 1️⃣ 读取 cleaned H1
h1_path = f"{base}/假设1数据/H1_cleaned.csv"
df = pd.read_csv(h1_path, encoding="utf-8-sig")

# 2️⃣ 读取 turnover 数据
trd_path = f"{base}/假设1数据/TRD_with_turnover.csv"
trd = pd.read_csv(trd_path, encoding="utf-8-sig")

# 3️⃣ 清理列名
df.columns = df.columns.str.strip()
trd.columns = trd.columns.str.strip()

# 4️⃣ 统一 key（非常关键）
df["code"] = df["code"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

trd["code"] = trd["Stkcd"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)
trd["year"] = pd.to_numeric(trd["Trdynt"], errors="coerce").astype("Int64")

# 5️⃣ 只保留需要的列（避免污染）
trd_y = trd[["code", "year", "turnover"]].copy()

# 6️⃣ merge 正确的 Y
df = pd.merge(
    df,
    trd_y,
    on=["code", "year"],
    how="left"
)

# 7️⃣ 检查 merge 结果
print("merge 后数据维度：", df.shape)
print("turnover 缺失数量：", df["turnover"].isna().sum())
print(df[["code", "year", "turnover"]].head())

# 8️⃣ 覆盖原文件（就是你要的）
df.to_csv(h1_path, index=False, encoding="utf-8-sig")

print("✅ H1 已更新：正确的 turnover 已加入")

merge 后数据维度： (21938, 25)
turnover 缺失数量： 0
     code  year  turnover
0  000002  2020  0.070691
1  000002  2021  0.110164
2  000002  2022  0.147576
3  000002  2023  0.177855
4  000002  2024  0.691726
✅ H1 已更新：正确的 turnover 已加入


In [64]:
import pandas as pd

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

# 1️⃣ 读取你的 H2/H3 数据（就是这个文件）
file_path = f"{base}/假设2数据/final_AI_Y_controls.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

# 2️⃣ 删除错误的 Y_return
if "Y_return" in df.columns:
    df = df.drop(columns=["Y_return"])
    print("已删除 Y_return")

# 3️⃣ 读取 turnover 数据
trd_path = f"{base}/假设1数据/TRD_with_turnover.csv"
trd = pd.read_csv(trd_path, encoding="utf-8-sig")

# 4️⃣ 清理列名
df.columns = df.columns.str.strip()
trd.columns = trd.columns.str.strip()

# 5️⃣ 统一 key（关键修正点）
df["code"] = df["code"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)
df["year"] = pd.to_numeric(df["report_year"], errors="coerce").astype("Int64")   # ✅ 用 report_year

trd["code"] = trd["Stkcd"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)
trd["year"] = pd.to_numeric(trd["Trdynt"], errors="coerce").astype("Int64")

# 6️⃣ 只保留 turnover
trd_y = trd[["code", "year", "turnover"]]

# 7️⃣ merge
df = pd.merge(df, trd_y, on=["code", "year"], how="left")

# 8️⃣ 检查
print(df[["code", "report_year", "turnover"]].head())
print("缺失数量:", df["turnover"].isna().sum())

# 9️⃣ 删除临时 year（可选，保持你原结构）
df = df.drop(columns=["year"])

# 🔟 覆盖原文件
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print("✅ 已完成：正确Y已替换")

已删除 Y_return
     code  report_year  turnover
0  000001         2022       NaN
1  000002         2019       NaN
2  000002         2020  0.070691
3  000002         2021  0.110164
4  000002         2022  0.147576
缺失数量: 76
✅ 已完成：正确Y已替换


In [66]:
import pandas as pd

# 读取数据
file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设2数据/final_AI_Y_controls.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

# 👉 只选关键变量（不要全放！）
variables = [
    "AI_innovation_disclosure",
    "AI_risk_disclosure",
    "turnover"
]

# 描述统计
desc = df[variables].describe().T

# 只保留老师要的
desc = desc[["count", "mean", "min", "max"]]

print(desc)

                          count      mean       min       max
AI_innovation_disclosure  399.0  0.998732  0.833333  1.000000
AI_risk_disclosure        399.0  0.001268  0.000000  0.166667
turnover                  324.0  0.587738  0.000378  3.982843


In [68]:
import pandas as pd

# 读取 H1 数据
file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据/H1_final_dataset_correct_Y.csv"
df = pd.read_csv(file_path, encoding="utf-8-sig")

# 👉 只选关键变量（不要全放）
variables = [
    "ai_disclosure_sentence_ratio",   # H1 的核心解释变量
    "turnover"         # 因变量
]

# 描述统计
desc = df[variables].describe().T

# 只保留老师要求的
desc = desc[["count", "mean", "min", "max"]]

print(desc)

                                count      mean      min        max
ai_disclosure_sentence_ratio  21938.0  0.003153  0.00000   0.115375
turnover                      21938.0  0.590021  0.00026  11.236048


In [38]:
import pandas as pd
import re

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"
path = f"{base}/假设2数据/ai_disclosure_sentences_FIXED.xlsx"

df = pd.read_excel(path)

text_col = "AI_sentence"  # 如果不对改一下

df[text_col] = df[text_col].astype(str)

# ================================
# 你的原始关键词（保持不动）
# ================================
keywords = [
    "人工智能", "AI", "AIGC", "生成式人工智能", "生成式AI",
    "机器学习", "深度学习", "神经网络", "大模型", "语言模型",
    "自然语言处理", "NLP", "计算机视觉", "机器视觉", "知识图谱",
    "智能化", "智慧化", "智能平台", "智能系统", "智能识别",
    "智能分析", "智能决策", "智能算法", "算法", "推荐算法",
    "预测模型", "数据模型", "数据挖掘", "数据驱动",
    "自动化", "自动识别", "自动生成", "自动控制",
    "云计算", "边缘计算", "算力", "GPU",
    "ChatGPT", "GPT", "多模态", "智能客服", "智能机器人", "机器人"
]

# ================================
# 判断“是否是假AI句子”
# ================================
def is_fake_ai(text):
    text_lower = text.lower()

    # 1. 包含 ai 字母
    if "ai" in text_lower:
        
        # 2. 但不包含“独立AI”
        if not re.search(r"(?<![A-Za-z])AI(?![A-Za-z])", text):
            
            # 3. 且不包含你原来的任何关键词
            if not any(k in text for k in keywords):
                return True
    
    return False

# ================================
# 删除假AI
# ================================
df_clean = df[~df[text_col].apply(is_fake_ai)].copy()

print("原始数量：", len(df))
print("删除假AI后：", len(df_clean))

# ================================
# 保存
# ================================
output_path = f"{base}/假设2数据/ai_sentences_remove_fake_AI.xlsx"
df_clean.to_excel(output_path, index=False)

print("已保存：", output_path)

原始数量： 5431
删除假AI后： 4872
已保存： /Users/snowiiy/Desktop/26 spring/MGS3001/假设2数据/ai_sentences_remove_fake_AI.xlsx


In [3]:
import os
import re
import time
import requests
import pandas as pd

# =========================
# 1. 只下载 000521
# =========================
stock_codes = ["000521"]

# 这里是披露年份
start_year = 2022
end_year = 2024

save_dir = "/Users/snowiiy/Desktop/26 spring/MGS3001/annual_reports"
os.makedirs(save_dir, exist_ok=True)

records_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/annual_report_download_records_000521.csv"

# =========================
# 2. 巨潮设置
# =========================
query_url = "http://www.cninfo.com.cn/new/hisAnnouncement/query"
download_base = "http://static.cninfo.com.cn/"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "http://www.cninfo.com.cn/",
    "X-Requested-With": "XMLHttpRequest"
}

def clean_title(title):
    title = re.sub(r"<.*?>", "", str(title))
    title = re.sub(r'[\\/:*?"<>|]', "_", title)
    return title

def get_market_info(code):
    if code.startswith(("0", "3")):
        column = "szse"
        plate = "sz"
        org_id = "gssz" + code.zfill(7)
    elif code.startswith("6"):
        column = "sse"
        plate = "sh"
        org_id = "gssh" + code.zfill(7)
    else:
        column = None
        plate = None
        org_id = None

    return column, plate, org_id

all_records = []

# =========================
# 3. 下载
# =========================
for code in stock_codes:
    column, plate, org_id = get_market_info(code)
    stock_param = f"{code},{org_id}"

    for year in range(start_year, end_year + 1):
        print(f"\nSearching {code} disclosure year {year}...")

        params = {
            "pageNum": 1,
            "pageSize": 30,
            "column": column,
            "tabName": "fulltext",
            "plate": plate,
            "stock": stock_param,
            "searchkey": "",
            "secid": "",
            "category": "category_ndbg_szsh;",
            "trade": "",
            "seDate": f"{year}-01-01~{year}-12-31",
            "sortName": "",
            "sortType": "",
            "isHLtitle": "true"
        }

        try:
            response = requests.post(
                query_url,
                headers=headers,
                data=params,
                timeout=20
            )

            print("Status:", response.status_code)

            if response.status_code != 200:
                print("Request failed.")
                continue

            data = response.json()
            announcements = data.get("announcements") or []
            print("Found:", len(announcements))

            downloaded = False

            for ann in announcements:
                title = clean_title(ann.get("announcementTitle", ""))
                adjunct_url = ann.get("adjunctUrl", "")

                if not adjunct_url:
                    continue

                if (
                    "年度报告" in title
                    and "摘要" not in title
                    and "英文" not in title
                    and "取消" not in title
                    and "更正" not in title
                    and "修订" not in title
                ):
                    pdf_url = download_base + adjunct_url
                    file_name = f"{code}_{year}_{title}.pdf"
                    file_path = os.path.join(save_dir, file_name)

                    if os.path.exists(file_path):
                        print("Already exists:", file_name)
                    else:
                        print("Downloading:", title)
                        pdf_response = requests.get(
                            pdf_url,
                            headers=headers,
                            timeout=30
                        )

                        if pdf_response.status_code == 200:
                            with open(file_path, "wb") as f:
                                f.write(pdf_response.content)
                            print("✅ Downloaded:", file_name)
                        else:
                            print("❌ PDF download failed:", pdf_response.status_code)
                            continue

                    all_records.append({
                        "code": code,
                        "disclosure_year": year,
                        "title": title,
                        "pdf_url": pdf_url,
                        "file_path": file_path
                    })

                    downloaded = True
                    break

            if not downloaded:
                print("No valid annual report downloaded.")

        except Exception as e:
            print(f"❌ Error for {code} {year}: {e}")

        time.sleep(1)

# =========================
# 4. 保存记录
# =========================
records = pd.DataFrame(all_records)
records.to_csv(records_path, index=False, encoding="utf-8-sig")

print("\nDone.")
print(records)
print("PDF folder:", save_dir)


Searching 000521 disclosure year 2022...
Status: 200
Found: 2
Already exists: 000521_2022_2021年年度报告.pdf

Searching 000521 disclosure year 2023...
Status: 200
Found: 2
Already exists: 000521_2023_2022年年度报告.pdf

Searching 000521 disclosure year 2024...
Status: 200
Found: 2
Already exists: 000521_2024_2023年年度报告.pdf

Done.
     code  disclosure_year      title  \
0  000521             2022  2021年年度报告   
1  000521             2023  2022年年度报告   
2  000521             2024  2023年年度报告   

                                             pdf_url  \
0  http://static.cninfo.com.cn/finalpage/2022-03-...   
1  http://static.cninfo.com.cn/finalpage/2023-03-...   
2  http://static.cninfo.com.cn/finalpage/2024-03-...   

                                           file_path  
0  /Users/snowiiy/Desktop/26 spring/MGS3001/annua...  
1  /Users/snowiiy/Desktop/26 spring/MGS3001/annua...  
2  /Users/snowiiy/Desktop/26 spring/MGS3001/annua...  
PDF folder: /Users/snowiiy/Desktop/26 spring/MGS3001/annual_reports


In [4]:
import os
import re
import pandas as pd
import pdfplumber

# ==================================================
# 1. 路径设置
# ==================================================

pdf_folder = "/Users/snowiiy/Desktop/26 spring/MGS3001/annual_reports"

sentence_file = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences.xlsx"
firm_year_file = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_firm_year.xlsx"

# 只处理 000521 这几个披露年份
target_code = "000521"
target_disclosure_years = ["2022", "2023", "2024"]

# ==================================================
# 2. AI keywords
# ==================================================

ai_keywords = [
    "人工智能", "AI", "AIGC", "生成式人工智能", "生成式AI",
    "机器学习", "深度学习", "神经网络", "大模型", "语言模型",
    "自然语言处理", "NLP", "计算机视觉", "机器视觉", "知识图谱",
    "智能化", "智慧化", "智能平台", "智能系统", "智能识别",
    "智能分析", "智能决策", "智能算法", "算法", "推荐算法",
    "预测模型", "数据模型", "数据挖掘", "数据驱动",
    "自动化", "自动识别", "自动生成", "自动控制",
    "云计算", "边缘计算", "算力", "GPU",
    "ChatGPT", "GPT", "多模态", "智能客服", "智能机器人", "机器人"
]

# ==================================================
# 3. 工具函数
# ==================================================

def extract_text_from_pdf(pdf_path):
    text = ""

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                try:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                except:
                    continue
    except Exception as e:
        print("无法读取PDF:", pdf_path)
        print(e)

    return text


def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", "", text)
    return text


def split_sentences(text):
    text = clean_text(text)
    sentences = re.split(r"[。！？!?；;]", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 8]
    return sentences


def find_keywords(sentence, keywords):
    sentence_lower = sentence.lower()
    matched = [kw for kw in keywords if kw.lower() in sentence_lower]
    return matched


def extract_code_year(file_name):
    # 文件名例子：000521_2023_2022年年度报告.pdf
    parts = file_name.split("_")

    code = parts[0] if len(parts) > 0 else ""
    disclosure_year = parts[1] if len(parts) > 1 else ""

    year_match = re.search(r"(\d{4})年年度报告", file_name)
    report_year = year_match.group(1) if year_match else disclosure_year

    return code, disclosure_year, report_year

# ==================================================
# 4. 找到 000521 的目标 PDF
# ==================================================

pdf_files = []

for f in os.listdir(pdf_folder):
    if not f.lower().endswith(".pdf"):
        continue

    code, disclosure_year, report_year = extract_code_year(f)

    if code == target_code and disclosure_year in target_disclosure_years:
        pdf_files.append(f)

print("找到目标PDF数量:", len(pdf_files))
print(pdf_files)

# ==================================================
# 5. 提取 000521 的 AI sentences
# ==================================================

new_records = []

for idx, file_name in enumerate(pdf_files, start=1):
    print(f"\nProcessing {idx}/{len(pdf_files)}: {file_name}")

    file_path = os.path.join(pdf_folder, file_name)

    code, disclosure_year, report_year = extract_code_year(file_name)

    text = extract_text_from_pdf(file_path)

    if len(text) < 100:
        print("文本太少，可能是扫描版PDF，跳过")
        continue

    sentences = split_sentences(text)
    total_sentences = len(sentences)

    ai_count = 0

    for sentence_id, sentence in enumerate(sentences, start=1):
        matched_keywords = find_keywords(sentence, ai_keywords)

        if matched_keywords:
            ai_count += 1

            new_records.append({
                "code": code,
                "disclosure_year": disclosure_year,
                "report_year": report_year,
                "file_name": file_name,
                "sentence_id": sentence_id,
                "AI_sentence": sentence,
                "matched_keywords": ",".join(matched_keywords),
                "total_sentences_in_report": total_sentences
            })

    print("AI sentences found:", ai_count)

df_new_sentences = pd.DataFrame(new_records)

print("\n新提取AI句子数量:", len(df_new_sentences))

# ==================================================
# 6. 追加到 ai_disclosure_sentences.xlsx
# ==================================================

if os.path.exists(sentence_file):
    df_old_sentences = pd.read_excel(sentence_file)
    df_all_sentences = pd.concat([df_old_sentences, df_new_sentences], ignore_index=True)
else:
    df_all_sentences = df_new_sentences.copy()

# 去重，避免重复运行导致重复追加
df_all_sentences = df_all_sentences.drop_duplicates(
    subset=["code", "disclosure_year", "report_year", "file_name", "sentence_id", "AI_sentence"],
    keep="last"
)

df_all_sentences.to_excel(sentence_file, index=False)

print("\n句子级文件已更新:")
print(sentence_file)
print("总AI句子数量:", len(df_all_sentences))

# ==================================================
# 7. 重新生成 firm-year AI disclosure，并保存到原文件
# ==================================================

firm_year = (
    df_all_sentences
    .groupby(["code", "report_year"])
    .agg(
        ai_sentence_count=("AI_sentence", "count"),
        total_sentences_in_report=("total_sentences_in_report", "max")
    )
    .reset_index()
)

firm_year["AI_Disclosure"] = (
    firm_year["ai_sentence_count"] / firm_year["total_sentences_in_report"]
)

firm_year.to_excel(firm_year_file, index=False)

print("\n公司-年份级文件已更新:")
print(firm_year_file)
print(firm_year[firm_year["code"] == target_code])

找到目标PDF数量: 3
['000521_2023_2022年年度报告.pdf', '000521_2024_2023年年度报告.pdf', '000521_2022_2021年年度报告.pdf']

Processing 1/3: 000521_2023_2022年年度报告.pdf
AI sentences found: 20

Processing 2/3: 000521_2024_2023年年度报告.pdf
AI sentences found: 47

Processing 3/3: 000521_2022_2021年年度报告.pdf
AI sentences found: 27

新提取AI句子数量: 94


/var/folders/6f/q_xr3_ys4y5b78sztpmq5__r0000gn/T/ipykernel_46057/962156366.py:170: UserWarning: Cell contents too long (35744), truncated to 32767 characters
  df_all_sentences.to_excel(sentence_file, index=False)



句子级文件已更新:
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences.xlsx
总AI句子数量: 5431

公司-年份级文件已更新:
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_firm_year.xlsx
       code report_year  ai_sentence_count  total_sentences_in_report  \
428  000521        2021                 27                       1776   
429  000521        2022                 20                       1677   
430  000521        2023                 47                       1657   

     AI_Disclosure  
428       0.015203  
429       0.011926  
430       0.028365  


In [6]:
import pandas as pd
import os
from datetime import datetime

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

sentence_file = f"{base}/ai_disclosure_sentences.xlsx"
firm_year_file = f"{base}/ai_disclosure_firm_year.xlsx"

sentence_out = f"{base}/ai_disclosure_sentences_FIXED.xlsx"
firm_year_out = f"{base}/ai_disclosure_firm_year_FIXED.xlsx"

# 1. 修复 sentence-level
df_sentence = pd.read_excel(sentence_file, dtype={"code": str})

df_sentence["code"] = (
    df_sentence["code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_sentence["report_year"] = df_sentence["report_year"].astype(str)

df_sentence = df_sentence.sort_values(
    by=["code", "report_year", "sentence_id"],
    ascending=True
)

df_sentence.to_excel(sentence_out, index=False)

# 2. 修复 firm-year
df_firm = pd.read_excel(firm_year_file, dtype={"code": str})

df_firm["code"] = (
    df_firm["code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_firm["report_year"] = df_firm["report_year"].astype(str)

df_firm = df_firm.sort_values(
    by=["code", "report_year"],
    ascending=True
)

df_firm.to_excel(firm_year_out, index=False)

print("完成！已生成新文件：")
print(sentence_out)
print(firm_year_out)

print("sentence 文件是否存在：", os.path.exists(sentence_out))
print("firm-year 文件是否存在：", os.path.exists(firm_year_out))

print("\n前几行检查：")
print(df_firm.head(10))

完成！已生成新文件：
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences_FIXED.xlsx
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_firm_year_FIXED.xlsx
sentence 文件是否存在： True
firm-year 文件是否存在： True

前几行检查：
     code report_year  ai_sentence_count  total_sentences_in_report  \
0  000001        2022                 90                       2335   
1  000002        2019                  7                       1809   
2  000002        2020                  9                       1949   
3  000002        2021                 18                       1969   
4  000002        2022                 11                       1870   
5  000002        2023                 12                       1899   
6  000006        2020                  1                       1023   
7  000007        2019                  1                       1295   
8  000007        2020                  1                       1320   
9  000007        2022                  1                       1276   

  

In [5]:
import pandas as pd

# =========================
# 1. 文件路径
# =========================

sentence_file = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences.xlsx"

firm_year_file = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_firm_year.xlsx"

# =========================
# 2. 修复 sentence-level 文件
# =========================

df_sentence = pd.read_excel(sentence_file)

# 补齐6位股票代码
df_sentence["code"] = (
    df_sentence["code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_sentence.to_excel(sentence_file, index=False)

print("sentence-level 文件修复完成")

# =========================
# 3. 修复 firm-year 文件
# =========================

df_firm = pd.read_excel(firm_year_file)

df_firm["code"] = (
    df_firm["code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_firm.to_excel(firm_year_file, index=False)

print("firm-year 文件修复完成")

sentence-level 文件修复完成
firm-year 文件修复完成


In [7]:
import pandas as pd
import re

# ==================================================
# 1. 读取之前导出的 AI sentence 文件
# ==================================================

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences.xlsx"

df = pd.read_excel(file_path)

print("数据读取成功")
print(df.columns)

# ==================================================
# 2. Innovation Dictionary
# ==================================================

innovation_keywords = [

    "创新", "技术创新", "自主研发", "研发", "研发投入",
    "研发能力", "技术突破", "技术升级", "创新能力",

    "应用", "落地", "部署", "实施", "推广",
    "智能化", "数字化", "数字化转型", "智能化转型",

    "效率提升", "提升效率", "降本增效", "优化",
    "改善", "增强", "提高", "自动化",

    "核心竞争力", "竞争优势", "市场竞争力",
    "业务创新", "产品创新", "服务创新",

    "智能平台", "智能系统", "智能决策",
    "机器学习", "深度学习", "大模型",
    "自然语言处理", "计算机视觉",

    "增长", "盈利能力", "运营效率",
    "业务协同", "价值创造", "赋能"
]

# ==================================================
# 3. Risk Dictionary
# ==================================================

risk_keywords = [

    "风险", "不确定性", "挑战", "困难",
    "压力", "波动",

    "技术风险", "技术缺陷", "模型风险",
    "算法偏差", "误判", "失败",

    "数据安全", "隐私", "隐私保护",
    "信息安全", "网络安全", "信息泄露",

    "监管", "监管风险", "合规",
    "合规风险", "法律风险",

    "投入较高", "成本增加",
    "回报不确定", "投资风险",

    "依赖", "外部依赖",
    "人才短缺", "竞争加剧",

    "伦理风险", "伦理",
    "失业", "替代人工"
]

# ==================================================
# 4. 计算关键词数量
# ==================================================

def count_keywords(text, keyword_list):

    if pd.isna(text):
        return 0

    score = 0

    for kw in keyword_list:
        score += len(re.findall(re.escape(kw), str(text)))

    return score

# ==================================================
# 5. 自动识别 AI sentence 列
# ==================================================

possible_cols = [
    "AI_sentence",
    "sentence",
    "ai_sentence",
    "AI Sentence"
]

text_col = None

for col in possible_cols:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    text_col = df.columns[0]

print("使用列：", text_col)

# ==================================================
# 6. 计算 innovation / risk score
# ==================================================

df["innovation_score"] = df[text_col].apply(
    lambda x: count_keywords(x, innovation_keywords)
)

df["risk_score"] = df[text_col].apply(
    lambda x: count_keywords(x, risk_keywords)
)

# ==================================================
# 7. 分类
# ==================================================

def classify(row):

    if row["innovation_score"] > row["risk_score"]:
        return "innovation"

    elif row["risk_score"] > row["innovation_score"]:
        return "risk"

    elif row["risk_score"] == 0 and row["innovation_score"] == 0:
        return "neutral"

    else:
        return "mixed"

df["label"] = df.apply(classify, axis=1)

# ==================================================
# 8. 导出结果
# ==================================================

output_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_classified.xlsx"

df.to_excel(output_path, index=False)

# ==================================================
# 9. 输出结果
# ==================================================

print("\n完成！")
print("\n分类统计：")
print(df["label"].value_counts())

print("\n文件已保存：")
print(output_path)

数据读取成功
Index(['code', 'disclosure_year', 'report_year', 'file_name', 'sentence_id',
       'AI_sentence', 'matched_keywords', 'total_sentences_in_report'],
      dtype='str')
使用列： AI_sentence

完成！

分类统计：
label
innovation    3750
neutral       1387
risk           209
mixed           85
Name: count, dtype: int64

文件已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_classified.xlsx


In [8]:
import pandas as pd
import re

# ==================================================
# 1. 读取 AI disclosure sentence 文件
# ==================================================

file_path = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_disclosure_sentences_FIXED.xlsx"

df = pd.read_excel(
    file_path,
    dtype={"code": str}
)

# ==================================================
# 2. 修复股票代码（保留前导0）
# ==================================================

df["code"] = (
    df["code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

# ==================================================
# 3. Innovation Dictionary
# ==================================================

innovation_keywords = [

    "创新", "技术创新", "自主研发", "研发", "研发投入",
    "研发能力", "技术突破", "技术升级", "创新能力",

    "应用", "落地", "部署", "实施", "推广",
    "智能化", "数字化", "数字化转型", "智能化转型",

    "效率提升", "提升效率", "降本增效", "优化",
    "改善", "增强", "提高", "自动化",

    "核心竞争力", "竞争优势", "市场竞争力",
    "业务创新", "产品创新", "服务创新",

    "智能平台", "智能系统", "智能决策",
    "机器学习", "深度学习", "大模型",
    "自然语言处理", "计算机视觉",

    "增长", "盈利能力", "运营效率",
    "业务协同", "价值创造", "赋能"
]

# ==================================================
# 4. Risk Dictionary
# ==================================================

risk_keywords = [

    "风险", "不确定性", "挑战", "困难",
    "压力", "波动",

    "技术风险", "技术缺陷", "模型风险",
    "算法偏差", "误判", "失败",

    "数据安全", "隐私", "隐私保护",
    "信息安全", "网络安全", "信息泄露",

    "监管", "监管风险", "合规",
    "合规风险", "法律风险",

    "投入较高", "成本增加",
    "回报不确定", "投资风险",

    "依赖", "外部依赖",
    "人才短缺", "竞争加剧",

    "伦理风险", "伦理",
    "失业", "替代人工"
]

# ==================================================
# 5. 自动识别 AI sentence 列
# ==================================================

possible_cols = [
    "AI_sentence",
    "sentence",
    "ai_sentence",
    "AI Sentence"
]

text_col = None

for col in possible_cols:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError("找不到 AI sentence 列")

print("使用列：", text_col)

# ==================================================
# 6. 关键词计数函数
# ==================================================

def count_keywords(text, keyword_list):

    if pd.isna(text):
        return 0

    text = str(text)

    score = 0

    for kw in keyword_list:
        score += len(re.findall(re.escape(kw), text))

    return score

# ==================================================
# 7. 计算 innovation / risk score
# ==================================================

df["innovation_score"] = df[text_col].apply(
    lambda x: count_keywords(x, innovation_keywords)
)

df["risk_score"] = df[text_col].apply(
    lambda x: count_keywords(x, risk_keywords)
)

# ==================================================
# 8. 分类
# ==================================================

def classify(row):

    if row["innovation_score"] > row["risk_score"]:
        return "innovation"

    elif row["risk_score"] > row["innovation_score"]:
        return "risk"

    elif row["innovation_score"] == 0 and row["risk_score"] == 0:
        return "neutral"

    else:
        return "mixed"

df["label"] = df.apply(classify, axis=1)

# ==================================================
# 9. 排序（股票代码顺序恢复）
# ==================================================

df["report_year"] = df["report_year"].astype(str)

df = df.sort_values(
    by=["code", "report_year", "sentence_id"],
    ascending=True
)

# ==================================================
# 10. 导出句子级分类文件
# ==================================================

sentence_output = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_classified_FIXED.xlsx"

df.to_excel(sentence_output, index=False)

print("\n句子级文件已保存：")
print(sentence_output)

# ==================================================
# 11. firm-year aggregation
# ==================================================

firm_year = (
    df.groupby(["code", "report_year"])
    .agg(
        total_ai_sentences=(text_col, "count"),
        innovation_sentences=("label", lambda x: (x == "innovation").sum()),
        risk_sentences=("label", lambda x: (x == "risk").sum()),
        neutral_sentences=("label", lambda x: (x == "neutral").sum()),
        mixed_sentences=("label", lambda x: (x == "mixed").sum()),
        total_sentences_in_report=("total_sentences_in_report", "max")
    )
    .reset_index()
)

# ==================================================
# 12. 构造 disclosure variables
# ==================================================

firm_year["AI_Disclosure"] = (
    firm_year["total_ai_sentences"]
    / firm_year["total_sentences_in_report"]
)

firm_year["AI_Innovation_Disclosure"] = (
    firm_year["innovation_sentences"]
    / firm_year["total_sentences_in_report"]
)

firm_year["AI_Risk_Disclosure"] = (
    firm_year["risk_sentences"]
    / firm_year["total_sentences_in_report"]
)

firm_year["AI_Innovation_Ratio"] = (
    firm_year["innovation_sentences"]
    / firm_year["total_ai_sentences"]
)

firm_year["AI_Risk_Ratio"] = (
    firm_year["risk_sentences"]
    / firm_year["total_ai_sentences"]
)

# ==================================================
# 13. firm-year 排序
# ==================================================

firm_year = firm_year.sort_values(
    by=["code", "report_year"],
    ascending=True
)

# ==================================================
# 14. 导出 firm-year 文件
# ==================================================

firm_output = "/Users/snowiiy/Desktop/26 spring/MGS3001/ai_classified_firm_year_FIXED.xlsx"

firm_year.to_excel(firm_output, index=False)

print("\nfirm-year 文件已保存：")
print(firm_output)

print("\n分类统计：")
print(df["label"].value_counts())

print("\n前几行：")
print(firm_year.head())

使用列： AI_sentence

句子级文件已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_classified_FIXED.xlsx

firm-year 文件已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/ai_classified_firm_year_FIXED.xlsx

分类统计：
label
innovation    3750
neutral       1387
risk           209
mixed           85
Name: count, dtype: int64

前几行：
     code report_year  total_ai_sentences  innovation_sentences  \
0  000001        2022                  90                    68   
1  000002        2019                   7                     3   
2  000002        2020                   9                     2   
3  000002        2021                  18                     9   
4  000002        2022                  11                     4   

   risk_sentences  neutral_sentences  mixed_sentences  \
0               7                 14                1   
1               0                  4                0   
2               1                  6                0   
3               1                  8                0   
4  

In [9]:
import pandas as pd

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

df = pd.read_excel(f"{base}/ai_classified_firm_year_FIXED.xlsx", dtype={"code": str})

df.to_csv(f"{base}/ai_classified_firm_year_FIXED.csv", index=False, encoding="utf-8-sig")

sample = df.head(1000)
sample.to_csv(f"{base}/ai_classified_firm_year_sample.csv", index=False, encoding="utf-8-sig")

print("完成")
print(df.shape)
print(df.head())

完成
(428, 13)
     code  report_year  total_ai_sentences  innovation_sentences  \
0  000001         2022                  90                    68   
1  000002         2019                   7                     3   
2  000002         2020                   9                     2   
3  000002         2021                  18                     9   
4  000002         2022                  11                     4   

   risk_sentences  neutral_sentences  mixed_sentences  \
0               7                 14                1   
1               0                  4                0   
2               1                  6                0   
3               1                  8                0   
4               2                  5                0   

   total_sentences_in_report  AI_Disclosure  AI_Innovation_Disclosure  \
0                       2335       0.038544                  0.029122   
1                       1809       0.003870                  0.001658   
2               

In [12]:
import pandas as pd

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

df = pd.read_excel(
    f"{base}/ai_classified_FIXED.xlsx",
    dtype={"code": str}
)

sample = df.head(1000)

sample.to_csv(
    f"{base}/ai_classified_sentences_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

print("完成")
print(sample.shape)

完成
(1000, 11)


In [14]:
import pandas as pd

# ==================================================
# 文件夹路径
# ==================================================

base = "/Users/snowiiy/Desktop/26 spring/MGS3001/AS3 data"

# ==================================================
# 1. 转换 firm-year 文件
# ==================================================

df_firm = pd.read_excel(
    f"{base}/ai_disclosure_firm_year_FIXED.xlsx",
    dtype={"code": str}
)

df_firm.to_csv(
    f"{base}/ai_disclosure_firm_year_FIXED.csv",
    index=False,
    encoding="utf-8-sig"
)

print("firm-year CSV 已导出")

# ==================================================
# 2. 转换 sentence-level 文件
# ==================================================

df_sentence = pd.read_excel(
    f"{base}/ai_disclosure_sentences_FIXED.xlsx",
    dtype={"code": str}
)

df_sentence.to_csv(
    f"{base}/ai_disclosure_sentences_FIXED.csv",
    index=False,
    encoding="utf-8-sig"
)

print("sentence-level CSV 已导出")

# ==================================================
# 完成提示
# ==================================================

print("\n全部完成")
print(f"文件位置：{base}")

firm-year CSV 已导出
sentence-level CSV 已导出

全部完成
文件位置：/Users/snowiiy/Desktop/26 spring/MGS3001/AS3 data


In [17]:
import pandas as pd

# ================================
# 1. 路径（改成你实际位置）
# ================================
control_path = r"/Users/snowiiy/Desktop/26 spring/MGS3001/常用控制变量24（已剔除金融STPT已缩尾）.dta"

# ================================
# 2. 读取 .dta 文件
# ================================
df_control = pd.read_stata(control_path)

print("✅ 控制变量读取成功")
print("数据维度：", df_control.shape)

# ================================
# 3. 查看变量名（非常关键）
# ================================
print("\n变量名如下：")
print(df_control.columns)


✅ 控制变量读取成功
数据维度： (62479, 84)

变量名如下：
Index(['证券代码', '证券简称', 'stkcd', 'year', '行业代码', 'Size', 'Lev', 'EM1', 'EM2',
       'DER', 'DLCR', 'ROA', 'ROE', 'GrossProfit', 'NetProfit', 'Liquid',
       'Quick', 'CashRatio', 'Cashflow', 'REC', 'INV', 'FIXED', 'Intangible',
       'Tangible', 'Growth', 'AssetGrowth', 'NetProfitGrowth', 'Loss', 'CTR1',
       'CTR2', 'ITR', 'CAP', 'CMIR', 'RCA', 'FL', 'OL', 'CL', '年个股回报率1',
       '年个股回报率2', 'Invest1', 'Invest2', 'Invest3', 'Invest4', 'Board', 'Indep',
       'Dual', 'Top1', 'Top5', 'Top10', 'Balance1', 'Balance2', 'Balance3',
       'Herfindahl3', 'Herfindahl5', 'Herfindahl10', 'Seperate', 'BM', 'PB',
       'TobinQ', 'ListAge', 'FirmAge', 'Dturn', 'INST', 'Mshare', 'Mfee',
       'ATO', 'Occupy', 'Big4', 'Opinion', 'AuditFee', 'Employee',
       'Executives', 'TMTAge', 'Female', 'TMTPay1', 'TMTPay2', 'Shares',
       'ChairHoldR', 'CEOHoldR', 'SOE', 'Province', 'City', 'Industry',
       '丘知数据分析会员之家'],
      dtype='str')


In [20]:
import pandas as pd

# ================================
# 1. 路径
# ================================
base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

xy_path = f"{base}/merged_CSMAR_AI_TRD_firm_year.csv"
control_path = f"{base}/常用控制变量24（已剔除金融STPT已缩尾）.dta"

output_path = f"{base}/final_dataset_with_controls.csv"

# ================================
# 2. 读取数据
# ================================
df_xy = pd.read_csv(xy_path, dtype={"code": str})
df_control = pd.read_stata(control_path)

print("XY数据：", df_xy.shape)
print("控制变量数据：", df_control.shape)

# ================================
# 3. 统一 merge key
# ================================

# 控制变量里股票代码叫 stkc
df_control["code"] = df_control["stkcd"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)

# 控制变量里年份叫 year
df_control["report_year"] = df_control["year"].astype(int).astype(str)

# XY 数据也统一格式
df_xy["code"] = df_xy["code"].astype(str).str.replace(".0", "", regex=False).str.zfill(6)
df_xy["report_year"] = df_xy["year"].astype(int).astype(str)

# ================================
# 4. 选择控制变量
# ================================
control_vars = [
    "code",
    "report_year",
    "Size",
    "Lev",
    "ROA",
    "Growth",
    "CashRatio",
    "TobinQ",
    "ListAge",
    "Board",
    "Indep",
    "Dual",
    "Top1",
    "SOE"
]

df_control_clean = df_control[control_vars].copy()

# ================================
# 5. Merge
# ================================
df_final = pd.merge(
    df_xy,
    df_control_clean,
    on=["code", "report_year"],
    how="left"
)

# ================================
# 6. 检查 merge 结果
# ================================
print("合并后数据：", df_final.shape)

print("\n控制变量缺失情况：")
print(df_final[["Size", "Lev", "ROA", "Growth", "CashRatio", "TobinQ"]].isna().sum())

# ================================
# 7. 保存最终数据
# ================================
df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n完成！最终文件已保存：")
print(output_path)

print("\n前几行：")
print(df_final.head())

XY数据： (21938, 18)
控制变量数据： (62479, 84)
合并后数据： (21938, 31)

控制变量缺失情况：
Size         714
Lev          714
ROA          714
Growth       725
CashRatio    714
TobinQ       719
dtype: int64

完成！最终文件已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/final_dataset_with_controls.csv

前几行：
     code  year    end_date  institution_id industry_code  \
0  000002  2020  2020-12-31          101775           K70   
1  000002  2021  2021-12-31          101775           K70   
2  000002  2022  2022-12-31          101775           K70   
3  000002  2023  2023-12-31          101775           K70   
4  000002  2024  2024-12-31          101775           K70   

   annual_report_ai_word_freq  annual_report_ai_sentence_freq  \
0                           5                               5   
1                           9                               9   
2                           6                               6   
3                           7                               7   
4                           9    

In [21]:
import pandas as pd

# ================================
# 1. 路径
# ================================
base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

input_path = f"{base}/final_dataset_with_controls.csv"
output_path = f"{base}/sample_1000.csv"

# ================================
# 2. 读取数据
# ================================
df = pd.read_csv(input_path, dtype={"code": str})

print("原始数据大小：", df.shape)

# ================================
# 3. 抽样（1000条）
# ================================
sample = df.sample(n=1000, random_state=42)

print("抽样数据大小：", sample.shape)

# ================================
# 4. 保存 CSV
# ================================
sample.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n已保存：")
print(output_path)

原始数据大小： (21938, 31)
抽样数据大小： (1000, 31)

已保存：
/Users/snowiiy/Desktop/26 spring/MGS3001/sample_1000.csv


In [32]:
import pandas as pd
import os

# ================================
# 1. 路径设置
# ================================
base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

ai_path = f"{base}/假设2数据/ai_classified_firm_year_FIXED.xlsx"
trd_path = f"{base}/假设1数据/TRD_Year.xlsx"
control_path = f"{base}/常用控制变量24（已剔除金融STPT已缩尾）.dta"

output_path = f"{base}/final_h2_h3_dataset.csv"

# ================================
# 2. 检查文件是否存在
# ================================
print("AI文件存在吗:", os.path.exists(ai_path))
print("TRD文件存在吗:", os.path.exists(trd_path))
print("控制变量文件存在吗:", os.path.exists(control_path))

# ================================
# 3. 读取数据
# ================================
df_ai = pd.read_excel(ai_path, dtype={"code": str})
df_trd = pd.read_excel(trd_path)
df_control = pd.read_stata(control_path)

print("AI数据:", df_ai.shape)
print("TRD数据:", df_trd.shape)
print("控制变量数据:", df_control.shape)

# ================================
# 4. 清理列名空格
# ================================
df_ai.columns = df_ai.columns.astype(str).str.strip()
df_trd.columns = df_trd.columns.astype(str).str.strip()
df_control.columns = df_control.columns.astype(str).str.strip()

# ================================
# 5. 处理 AI 数据 key
# ================================
df_ai["code"] = (
    df_ai["code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_ai["report_year"] = (
    pd.to_numeric(df_ai["report_year"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

# ================================
# 6. 处理 TRD 数据 key 和 Y变量
# TRD真实列名：
# Stkcd = 股票代码
# Trdynt = 交易年份
# Yretnd = 年个股回报率
# ================================
df_trd["code"] = (
    df_trd["Stkcd"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_trd["report_year"] = (
    pd.to_numeric(df_trd["Trdynt"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

df_trd["Y_return"] = pd.to_numeric(df_trd["Yretnd"], errors="coerce")

df_trd_clean = df_trd[["code", "report_year", "Y_return"]].copy()

# 去掉无效年份
df_trd_clean = df_trd_clean[df_trd_clean["report_year"] != "<NA>"]

# ================================
# 7. 处理控制变量 key
# 控制变量真实列名：
# stkcd = 股票代码
# year = 年份
# ================================
df_control["code"] = (
    df_control["stkcd"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(6)
)

df_control["report_year"] = (
    pd.to_numeric(df_control["year"], errors="coerce")
    .astype("Int64")
    .astype(str)
)

# 选择控制变量
control_vars = [
    "code", "report_year",
    "Size", "Lev", "ROA", "Growth", "CashRatio",
    "TobinQ", "ListAge", "Board", "Indep",
    "Dual", "Top1", "SOE"
]

df_control_clean = df_control[control_vars].copy()
df_control_clean = df_control_clean[df_control_clean["report_year"] != "<NA>"]

# ================================
# 8. Merge: AI + TRD
# ================================
df_merge = pd.merge(
    df_ai,
    df_trd_clean,
    on=["code", "report_year"],
    how="left"
)

print("AI + TRD 合并后:", df_merge.shape)
print("Y_return 缺失数量:", df_merge["Y_return"].isna().sum())

# ================================
# 9. Merge: 再加控制变量
# ================================
df_final = pd.merge(
    df_merge,
    df_control_clean,
    on=["code", "report_year"],
    how="left"
)

print("最终合并后:", df_final.shape)

print("\n控制变量缺失情况:")
print(df_final[[
    "Size", "Lev", "ROA", "Growth", "CashRatio",
    "TobinQ", "ListAge", "Board", "Indep",
    "Dual", "Top1", "SOE"
]].isna().sum())

# ================================
# 10. 保存最终数据
# ================================
df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n完成！最终文件已保存:")
print(output_path)

print("\n前几行:")
print(df_final.head())

AI文件存在吗: True
TRD文件存在吗: True
控制变量文件存在吗: True


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


AI数据: (428, 13)
TRD数据: (22621, 6)
控制变量数据: (62479, 84)
AI + TRD 合并后: (428, 14)
Y_return 缺失数量: 84
最终合并后: (428, 26)

控制变量缺失情况:
Size         14
Lev          14
ROA          14
Growth       14
CashRatio    14
TobinQ       15
ListAge      14
Board        14
Indep        14
Dual         14
Top1         14
SOE          14
dtype: int64

完成！最终文件已保存:
/Users/snowiiy/Desktop/26 spring/MGS3001/final_h2_h3_dataset.csv

前几行:
     code report_year  total_ai_sentences  innovation_sentences  \
0  000001        2022                  90                    68   
1  000002        2019                   7                     3   
2  000002        2020                   9                     2   
3  000002        2021                  18                     9   
4  000002        2022                  11                     4   

   risk_sentences  neutral_sentences  mixed_sentences  \
0               7                 14                1   
1               0                  4                0   
2            

In [33]:
import pandas as pd

base = "/Users/snowiiy/Desktop/26 spring/MGS3001"

# H2/H3 final dataset
df = pd.read_csv(f"{base}/final_h2_h3_dataset.csv", dtype={"code": str})

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns)

# 选择关键数值变量
numeric_vars = [
    "Y_return",
    "total_ai_sentences",
    "innovation_sentences",
    "risk_sentences",
    "neutral_sentences",
    "mixed_sentences",
    "AI_Disclosure",
    "AI_Innovation_Disclosure",
    "AI_Risk_Disclosure",
    "Size",
    "Lev",
    "ROA",
    "Growth",
    "CashRatio",
    "TobinQ",
    "ListAge",
    "Board",
    "Indep",
    "Dual",
    "Top1",
    "SOE"
]

# 只保留真实存在的列
numeric_vars = [v for v in numeric_vars if v in df.columns]

desc = df[numeric_vars].describe().T

# 只保留老师要求的 count, mean, min, max
desc_simple = desc[["count", "mean", "min", "max"]]

print(desc_simple)

# 导出
desc_simple.to_csv(f"{base}/descriptive_statistics_h2_h3.csv", encoding="utf-8-sig")

print("Descriptive statistics saved.")

Dataset shape: (428, 26)
Columns:
Index(['code', 'report_year', 'total_ai_sentences', 'innovation_sentences',
       'risk_sentences', 'neutral_sentences', 'mixed_sentences',
       'total_sentences_in_report', 'AI_Disclosure',
       'AI_Innovation_Disclosure', 'AI_Risk_Disclosure', 'AI_Innovation_Ratio',
       'AI_Risk_Ratio', 'Y_return', 'Size', 'Lev', 'ROA', 'Growth',
       'CashRatio', 'TobinQ', 'ListAge', 'Board', 'Indep', 'Dual', 'Top1',
       'SOE'],
      dtype='str')
                          count       mean        min         max
Y_return                  344.0   0.074830  -0.511682    5.659375
total_ai_sentences        428.0  12.689252   1.000000  149.000000
innovation_sentences      428.0   8.761682   0.000000   82.000000
risk_sentences            428.0   0.488318   0.000000    7.000000
neutral_sentences         428.0   3.240654   0.000000   70.000000
mixed_sentences           428.0   0.198598   0.000000    5.000000
AI_Disclosure             428.0   0.008670   0.000478

In [34]:
import pandas as pd

base = "/Users/snowiiy/Desktop/26 spring/MGS3001/假设1数据"

df = pd.read_csv(f"{base}/final_dataset_with_controls.csv", dtype={"code": str})

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns)

numeric_vars = [
    "investor_sentiment_proxy_log_trading_volume",
    "annual_report_ai_word_freq",
    "annual_report_ai_sentence_freq",
    "ai_disclosure_sentence_ratio",
    "ai_disclosure_word_ratio",
    "Size",
    "Lev",
    "ROA",
    "Growth",
    "CashRatio",
    "TobinQ",
    "ListAge",
    "Board",
    "Indep",
    "Dual",
    "Top1",
    "SOE"
]

numeric_vars = [v for v in numeric_vars if v in df.columns]

desc = df[numeric_vars].describe().T
desc_simple = desc[["count", "mean", "min", "max"]]

print(desc_simple)

desc_simple.to_csv(f"{base}/descriptive_statistics_h1.csv", encoding="utf-8-sig")

print("H1 descriptive statistics saved.")

Dataset shape: (21938, 31)
Columns:
Index(['code', 'year', 'end_date', 'institution_id', 'industry_code',
       'annual_report_ai_word_freq', 'annual_report_ai_sentence_freq',
       'annual_report_total_sentences', 'annual_report_total_words',
       'ai_disclosure_sentence_ratio', 'ai_disclosure_word_ratio',
       'ai_sentence_per_1000_sentences', 'ai_word_per_1000_words',
       'annual_trading_shares', 'float_market_value_thousand',
       'total_market_value_thousand', 'annual_return_no_cash_dividend',
       'investor_sentiment_proxy_log_trading_volume', 'report_year', 'Size',
       'Lev', 'ROA', 'Growth', 'CashRatio', 'TobinQ', 'ListAge', 'Board',
       'Indep', 'Dual', 'Top1', 'SOE'],
      dtype='str')
                                               count       mean        min  \
investor_sentiment_proxy_log_trading_volume  21938.0  21.230742  13.304520   
annual_report_ai_word_freq                   21938.0  10.884037   0.000000   
annual_report_ai_sentence_freq           